In [1]:
import tensorflow as tf
from tensorflow.keras.datasets import fashion_mnist

(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

print("Training images:", x_train.shape)
print("Test images:", x_test.shape)

Training images: (60000, 28, 28)
Test images: (10000, 28, 28)


In [2]:
x_validation = x_train[-5000:]
y_validation = y_train[-5000:]

x_train = x_train[:-5000]
y_train = y_train[:-5000]

image_size = (96, 96)
batch_size = 64


def prepare_image(image, label):
    image = tf.expand_dims(image, axis=-1)
    image = tf.image.resize(image, image_size)
    image = tf.image.grayscale_to_rgb(image)
    image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
    return image, label


train_dataset = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(10000)
    .map(prepare_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

validation_dataset = (
    tf.data.Dataset.from_tensor_slices((x_validation, y_validation))
    .map(prepare_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

test_dataset = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(prepare_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

print("MobileNetV2 data pipeline is ready.")

MobileNetV2 data pipeline is ready.


In [3]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(96, 96, 3),
    include_top=False,
    weights="imagenet",
)

base_model.trainable = False

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(96, 96, 3)),
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(10, activation="softmax"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 8s 1us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ random_flip (RandomFlip)        │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation                 │ (None, 96, 96, 3)      │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_96             │ (None, 3, 3, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,270,794 (8.66 MB)

 Trainable params: 12,810 (50.04 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [4]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True,
    )
]

history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=3,
    callbacks=callbacks,
)

Epoch 1/3


/Users/anujanag/Documents/smart-retail-ai/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


860/860 ━━━━━━━━━━━━━━━━━━━━ 96s 109ms/step - accuracy: 0.7969 - loss: 0.5764 - val_accuracy: 0.8642 - val_loss: 0.3881
Epoch 2/3
860/860 ━━━━━━━━━━━━━━━━━━━━ 100s 116ms/step - accuracy: 0.8446 - loss: 0.4396 - val_accuracy: 0.8770 - val_loss: 0.3586
Epoch 3/3
860/860 ━━━━━━━━━━━━━━━━━━━━ 100s 116ms/step - accuracy: 0.8507 - loss: 0.4147 - val_accuracy: 0.8770 - val_loss: 0.3559


In [5]:
test_loss, test_accuracy = model.evaluate(test_dataset)

print("MobileNetV2 test accuracy:", round(test_accuracy * 100, 2), "%")

157/157 ━━━━━━━━━━━━━━━━━━━━ 16s 99ms/step - accuracy: 0.8622 - loss: 0.3920
MobileNetV2 test accuracy: 86.22 %


In [6]:
base_model.trainable = True

for layer in base_model.layers[:100]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

fine_tune_history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=2,
    callbacks=callbacks,
)

Epoch 1/2
860/860 ━━━━━━━━━━━━━━━━━━━━ 172s 194ms/step - accuracy: 0.7723 - loss: 0.7286 - val_accuracy: 0.8716 - val_loss: 0.3952
Epoch 2/2
860/860 ━━━━━━━━━━━━━━━━━━━━ 174s 202ms/step - accuracy: 0.8414 - loss: 0.4692 - val_accuracy: 0.8784 - val_loss: 0.3594


In [7]:
test_loss, test_accuracy = model.evaluate(test_dataset)

print("Fine-tuned MobileNetV2 test accuracy:", round(test_accuracy * 100, 2), "%")

157/157 ━━━━━━━━━━━━━━━━━━━━ 15s 92ms/step - accuracy: 0.8679 - loss: 0.3924
Fine-tuned MobileNetV2 test accuracy: 86.79 %


In [8]:
model.save("../app/models/product_classifier.h5")

print("Final MobileNetV2 model saved.")


Final MobileNetV2 model saved.
